# PNAD dataset generation

This notebook generates one refined PNAD dataset per available survey year. All year-specific ingestion rules are read from `df_metadata.xlsx`, generated by `00_cria_metadata.ipynb`.

The metadata defines field positions and widths, household-size information, missing-income sentinels, raw-file patterns, subdirectories, and expected file counts. The code below therefore contains no year-specific PNAD ingestion rules.


In [1]:
from pathlib import Path
from time import perf_counter

import pandas as pd
from tqdm.auto import tqdm


## Paths and metadata

The metadata file is read from the fixed project location. The raw and refined data directories are derived from the same project root.


In [2]:
METADATA_PATH = Path(
    r"C:\Users\Osvaldo\OneDrive\academic_research\econophysics\projeto_pnad_ic_beatriz\metadata\df_metadata.xlsx"
)

PROJECT_ROOT = METADATA_PATH.parent.parent
RAW_PATH = PROJECT_ROOT / "dados_raw"
REFINED_PATH = PROJECT_ROOT / "dados_refined"

if not METADATA_PATH.is_file():
    raise FileNotFoundError(
        f"Metadata não encontrado: {METADATA_PATH}. Execute 00_cria_metadata.ipynb primeiro."
    )

if not RAW_PATH.is_dir():
    raise FileNotFoundError(f"Pasta de dados brutos não encontrada: {RAW_PATH}")

REFINED_PATH.mkdir(parents=True, exist_ok=True)

df_metadata = pd.read_excel(METADATA_PATH, dtype={"raw_subdir": "string"})

required_columns = {
    "ano",
    "pos_renda",
    "tam_renda",
    "pos_morador",
    "tam_morador",
    "missing_renda",
    "raw_subdir",
    "raw_pattern",
    "n_files",
}
missing_columns = required_columns.difference(df_metadata.columns)
if missing_columns:
    raise ValueError(
        "Metadata incompleto. Colunas ausentes: "
        + ", ".join(sorted(missing_columns))
    )

assert df_metadata["ano"].is_unique, "Metadata contém anos duplicados."

df_metadata.head()


,ano,var_renda,pos_renda,tam_renda,var_morador,pos_morador,tam_morador,link,raw_subdir,raw_pattern,n_files,missing_renda,Currency,Exchange,Index,Adjust2025,Inflation
0,1976,V2954,227.0,9.0,NaN,NaN,NaN,https://ftp.ibge.gov.br/Trabalho_e_Rendimento/...,<NA>,DOM1976.*,1,9999999.0,cruzeiro-Cr$,11.313,100.00000,4.629253,5.629253
1,1977,V131,288.0,9.0,NaN,NaN,NaN,https://ftp.ibge.gov.br/Trabalho_e_Rendimento/...,<NA>,DOM1977.*,1,999999999.0,cruzeiro-Cr$,14.930,106.42361,4.289478,5.289478
2,1978,V2541,214.0,9.0,NaN,NaN,NaN,https://ftp.ibge.gov.br/Trabalho_e_Rendimento/...,<NA>,DOM1978.*,1,999999999.0,cruzeiro-Cr$,19.050,115.45139,3.875865,4.875865
3,1979,V2517,167.0,9.0,NaN,NaN,NaN,https://ftp.ibge.gov.br/Trabalho_e_Rendimento/...,<NA>,DOM1979.*,1,999999999.0,cruzeiro-Cr$,28.792,129.16667,3.358132,4.358132
4,1980,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,NaN,0,NaN,cruzeiro-Cr$,NaN,NaN,NaN,NaN


## Fast fixed-width ingestion

Only the required fixed-width fields are extracted directly from the raw byte lines. This avoids the overhead of `pandas.read_fwf()` while preserving the same zero-based, half-open field positions stored in the metadata. Missing-income filtering and the per-capita transformation are applied during ingestion, reducing memory use.


In [3]:
def get_source_files(spec):
    folder = RAW_PATH

    if pd.notna(spec.raw_subdir):
        raw_subdir = str(spec.raw_subdir).strip()
        if raw_subdir:
            folder = folder / raw_subdir

    if not folder.is_dir():
        raise FileNotFoundError(
            f"Pasta de dados brutos não encontrada para {int(spec.ano)}: {folder}"
        )

    pattern = str(spec.raw_pattern).strip()
    files = sorted(p for p in folder.glob(pattern) if p.is_file())

    expected = int(spec.n_files)
    if len(files) != expected:
        raise RuntimeError(
            f"Esperados {expected} arquivo(s) para {int(spec.ano)}, "
            f"encontrados {len(files)} em {folder} com padrão '{pattern}': "
            + ", ".join(p.name for p in files)
        )

    return files


def _parse_int_field(line, start, end):
    field = line[start:end].strip()
    if not field:
        return None

    try:
        return int(field)
    except ValueError:
        return None


def read_year_fast(spec, show_file_progress=True):
    year = int(spec.ano)
    files = get_source_files(spec)

    income_start = int(spec.pos_renda)
    income_end = income_start + int(spec.tam_renda)
    missing_income = int(spec.missing_renda)

    has_member = pd.notna(spec.pos_morador) and pd.notna(spec.tam_morador)
    if has_member:
        member_start = int(spec.pos_morador)
        member_end = member_start + int(spec.tam_morador)

    incomes = []
    n_raw = 0
    n_missing_income = 0
    n_invalid_income = 0
    n_invalid_member = 0

    for file in files:
        file_size = file.stat().st_size

        progress = tqdm(
            total=file_size,
            desc=f"{year} | {file.name}",
            unit="B",
            unit_scale=True,
            leave=False,
            disable=not show_file_progress,
        )

        bytes_since_update = 0

        with file.open("rb", buffering=1024 * 1024) as fh:
            for line in fh:
                n_raw += 1
                bytes_since_update += len(line)

                income = _parse_int_field(line, income_start, income_end)

                if income is None:
                    n_invalid_income += 1
                elif income == missing_income:
                    n_missing_income += 1
                elif has_member:
                    member = _parse_int_field(line, member_start, member_end)

                    if member is None or member <= 0:
                        n_invalid_member += 1
                    else:
                        incomes.append(income / member)
                else:
                    incomes.append(float(income))

                # Updating tqdm on every line is unnecessarily expensive.
                if bytes_since_update >= 4 * 1024 * 1024:
                    progress.update(bytes_since_update)
                    bytes_since_update = 0

        if bytes_since_update:
            progress.update(bytes_since_update)

        progress.close()

    df = pd.DataFrame({
        "renda": pd.Series(incomes, dtype="float64"),
        "ano": year,
    })

    stats = {
        "ano": year,
        "arquivos": len(files),
        "n_raw": n_raw,
        "n_refined": len(df),
        "n_missing_renda": n_missing_income,
        "n_invalid_renda": n_invalid_income,
        "n_invalid_morador": n_invalid_member if has_member else 0,
        "per_capita": has_member,
    }

    return df, stats


## Generate refined datasets

Each year is read, filtered, transformed and immediately written to Parquet. The outer progress bar tracks survey years and the temporary inner bar tracks bytes read from the current raw file. The summary table records retained and discarded observations.


In [4]:
summary = []

available = df_metadata.loc[
    df_metadata["pos_renda"].notna()
    & df_metadata["tam_renda"].notna()
    & df_metadata["raw_pattern"].notna()
    & (df_metadata["raw_pattern"].astype(str).str.strip() != "")
    & (df_metadata["n_files"].fillna(0) > 0)
    & df_metadata["missing_renda"].notna()
].copy()

available = available.sort_values("ano")

years_progress = tqdm(
    available.itertuples(index=False),
    total=len(available),
    desc="PNAD",
    unit="ano",
)

for spec in years_progress:
    year = int(spec.ano)
    years_progress.set_postfix_str(str(year))

    t0 = perf_counter()

    df_refined, stats = read_year_fast(spec, show_file_progress=True)

    if df_refined.empty:
        raise RuntimeError(f"Dataset refinado vazio para {year}.")

    if df_refined["renda"].isna().any():
        raise RuntimeError(f"Valores NaN em renda após processamento de {year}.")

    if not (df_refined["ano"] == year).all():
        raise RuntimeError(f"Coluna ano inconsistente em {year}.")

    output_file = REFINED_PATH / f"pnad_refined_{year}.parquet"
    df_refined.to_parquet(
        output_file,
        index=False,
        compression="snappy",
    )

    stats["tempo_s"] = perf_counter() - t0
    stats["output"] = output_file.name
    summary.append(stats)

    del df_refined

df_summary = pd.DataFrame(summary)

assert len(df_summary) == len(available)
assert df_summary["ano"].is_unique

df_summary


PNAD:   0%|          | 0/45 [00:00<?, ?ano/s]

1976 | DOM1976.txt:   0%|          | 0.00/175M [00:00<?, ?B/s]

1977 | DOM1977.txt:   0%|          | 0.00/195M [00:00<?, ?B/s]

1978 | DOM1978.txt:   0%|          | 0.00/192M [00:00<?, ?B/s]

1979 | DOM1979.txt:   0%|          | 0.00/111M [00:00<?, ?B/s]

...

2012 | DOM2012.txt:   0%|          | 0.00/29.6M [00:00<?, ?B/s]

2013 | DOM2013.txt:   0%|          | 0.00/35.7M [00:00<?, ?B/s]

2014 | DOM2014.txt:   0%|          | 0.00/36.0M [00:00<?, ?B/s]

2015 | DOM2015.txt:   0%|          | 0.00/36.7M [00:00<?, ?B/s]

RuntimeError: Esperados 1 arquivo(s) para 2016, encontrados 0 em C:\Users\Osvaldo\OneDrive\academic_research\econophysics\projeto_pnad_ic_beatriz\dados_raw com padrão 'DOM2016.*': 